In [1]:
from libraries import *
from parameters import *
from util import *
import pandas as pd
from matplotlib import pyplot as plt
from matplotlib_venn import venn2

import scipy.sparse as sp
from sklearn.metrics.pairwise import cosine_distances
from sklearn.metrics import pairwise_distances


/home/beraslan/miniconda/envs/py312/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
1/30

0.03333333333333333

In [4]:
ed_df=pd.read_csv("DrugEnergyDistances.csv", index_col=0)
np.array(ed_df)

array([[3.51524353e-03, 1.43167229e+01, 6.83038597e+01, 6.41029968e+01,
        2.05006905e+01, 1.21805077e+01, 2.04711704e+01, 2.83905411e+01,
        1.43120060e+01, 3.28639984e+01, 2.38128033e+01, 1.93308029e+01,
        2.02079830e+01, 1.96421204e+01],
       [1.43167229e+01, 3.99017334e-03, 6.41949692e+01, 5.88757000e+01,
        9.11648178e+00, 6.09707451e+00, 1.39560452e+01, 1.67984982e+01,
        2.10213470e+00, 1.81862698e+01, 1.43539448e+01, 9.94906044e+00,
        5.36389732e+00, 5.18603897e+00],
       [6.83038597e+01, 6.41949692e+01, 3.48854065e-03, 1.18985958e+01,
        6.18402863e+01, 6.37742729e+01, 6.21859264e+01, 6.50830612e+01,
        6.61706276e+01, 5.51019363e+01, 7.16519051e+01, 5.90398598e+01,
        6.89816628e+01, 6.97982903e+01],
       [6.41029968e+01, 5.88757000e+01, 1.18985958e+01, 3.12805176e-03,
        6.39441166e+01, 5.96816368e+01, 6.46134262e+01, 6.80436554e+01,
        6.04623165e+01, 5.90507107e+01, 6.66075554e+01, 6.17370968e+01,
        6.373

In [ ]:
def energy_distance_multivariate(X, Y, metric: str = "euclidean") -> float:
    """
    Multivariate energy distance between two samples X and Y.

    X: (n_x, d), Y: (n_y, d)
    Each row is a d-dimensional observation (here: a cell's gene vector).
    """
    X = np.asarray(X)
    Y = np.asarray(Y)

    D_xy = pairwise_distances(X, Y, metric=metric)
    D_xx = pairwise_distances(X, X, metric=metric)
    D_yy = pairwise_distances(Y, Y, metric=metric)

    return 2.0 * D_xy.mean() - D_xx.mean() - D_yy.mean()


In [ ]:
adata_all=sc.read_h5ad("/processed_datasets/VCI/ChemoGenetic_H1_Basak/All_control_scaled_subset.h5ad")

In [ ]:
total_var = adata_all.uns["pca"]["variance_ratio"][:100].sum()
print(total_var)

In [ ]:
sample_key = "sample",
out_prefix = "sample_distances_v2",
max_cells_per_sample= 10000,
within_reps = 10,
n_perm= 100

In [ ]:
X = adata_all.obsm["X_pca"]


In [ ]:
adata_all.obs

In [ ]:
samples = adata_all.obs["sample"].astype("string")
unique_samples = np.array(sorted(samples.unique()))
n_samples = len(unique_samples)
n_samples


In [ ]:
sample_mats = {}
sample_means = []

for s in unique_samples:
    mask = (samples == s).values
    idx_all = np.where(mask)[0]
    if idx_all.size == 0:
        raise ValueError(f"No cells found for sample {s!r}.")

    # Optional subsampling for energy distance
    idx = idx_all
    X_sub = X[idx]

    # to dense
    if sp.issparse(X_sub):
        X_sub = X_sub.toarray()
    else:
        X_sub = np.asarray(X_sub)

    sample_mats[s] = X_sub

    # centroid for cosine distance (use *all* cells of that sample)
    X_all = X[idx_all]
    if sp.issparse(X_all):
        mean_vec = np.asarray(X_all.mean(axis=0)).ravel()
    else:
        mean_vec = np.asarray(X_all).mean(axis=0)
    sample_means.append(mean_vec)

sample_means = np.vstack(sample_means)  # (n_samples, n_genes)


In [ ]:
cos_mat = cosine_distances(sample_means, sample_means)
cos_df = pd.DataFrame(cos_mat, index=unique_samples, columns=unique_samples)


In [ ]:
import numpy as np
import pandas as pd

n_reps = 10
rng = np.random.default_rng(0)  # reproducible

ed_mat = np.zeros((n_samples, n_samples), dtype=float)

for i in range(n_samples):
    print("iiiii")
    print(i)
    s_i = unique_samples[i]
    Xi = sample_mats[s_i]
    ni = Xi.shape[0]
    mi = max(2, ni // 2)

    # ---------- diagonal: within-sample baseline ----------
    eds_diag = np.empty(n_reps, dtype=float)
    for r in range(n_reps):
        perm = rng.permutation(ni)
        idx1 = perm[:mi]
        idx2 = perm[mi:mi + mi]  # second half
        eds_diag[r] = energy_distance_multivariate(
            Xi[idx1], Xi[idx2], metric="euclidean"
        )
    ed_mat[i, i] = float(np.median(eds_diag))

    # ---------- off-diagonal ----------
    for j in range(i + 1, n_samples):
        print("jjjjjj")
        print(j)
        s_j = unique_samples[j]
        Xj = sample_mats[s_j]
        nj = Xj.shape[0]
        mj = max(2, nj // 2)

        eds = np.empty(n_reps, dtype=float)
        for r in range(n_reps):
            idx_i = rng.choice(ni, size=mi, replace=False)
            idx_j = rng.choice(nj, size=mj, replace=False)

            eds[r] = energy_distance_multivariate(
                Xi[idx_i], Xj[idx_j], metric="euclidean"
            )

        ed = float(np.median(eds))
        ed_mat[i, j] = ed
        ed_mat[j, i] = ed

ed_df = pd.DataFrame(ed_mat, index=unique_samples, columns=unique_samples)


In [ ]:

def compute_sample_distance_matrices_multivariate(
    adata,
    sample_key: str = "sample",
    layer: str | None = None,
    out_prefix: str = "sample_distances",
    max_cells_per_sample: int | None = None,
    random_state: int = 0,
    metric: str = "euclidean",
    # NEW:
    within_reps: int = 5,
    n_perm: int = 0,
):
    """
    Compute pairwise cosine and multivariate energy distances between samples,
    plus within-sample energy distances and optional permutation-based
    significance testing of between-sample distances.

    - Cosine distances are computed between sample mean expression vectors.
    - Energy distances are computed between the full *cell-level distributions*,
      treating each cell as a d-dimensional vector of gene expression.
    - Within-sample energy distance is estimated by splitting each sample into
      two random halves (repeated `within_reps` times) and computing the
      energy distance between them.
    - If `n_perm > 0`, between-sample energy distances are tested for
      significance by permutation: pooling cells from both samples and
      repeatedly reassigning them to groups of equal size.

    Parameters
    ----------
    adata : anndata.AnnData
        AnnData with cells as observations and genes as variables.
    sample_key : str
        Column in `adata.obs` giving the sample ID for each cell.
    layer : str or None
        If not None, use `adata.layers[layer]` instead of `adata.X`.
    out_prefix : str
        Prefix for output CSVs:
          - f"{out_prefix}_cosine.csv"
          - f"{out_prefix}_energy.csv"
          - f"{out_prefix}_energy_within.csv"
          - f"{out_prefix}_energy_pvals.csv" (if n_perm > 0)
    max_cells_per_sample : int or None
        Optional cap on number of cells per sample for energy distance
        (subsampling without replacement for speed).
    random_state : int
        RNG seed for reproducibility of subsampling and permutations.
    metric : str
        Distance metric for multivariate energy distance (default "euclidean").
    within_reps : int
        Number of random splits per sample to estimate within-sample
        energy distance (average across reps).
    n_perm : int
        Number of permutations for significance testing of between-sample
        energy distances. If 0, skip permutation testing.

    Returns
    -------
    cos_df : pd.DataFrame
        Sample × sample cosine distance matrix (centroids).
    ed_df : pd.DataFrame
        Sample × sample multivariate energy distance matrix.
    ed_within_df : pd.DataFrame
        DataFrame with one row per sample and a column "energy_within"
        with within-sample energy distance.
    ed_pval_df : pd.DataFrame or None
        Sample × sample matrix of permutation p-values for energy distance,
        or None if n_perm == 0.
    """
    rng = np.random.default_rng(random_state)

    # -------- 1. choose matrix ----------
    X = adata.layers[layer] if layer is not None else adata.X

    # -------- 2. sample labels ----------
    if sample_key not in adata.obs:
        raise KeyError(f"{sample_key!r} not found in adata.obs")
    samples = adata.obs[sample_key].astype("string")
    unique_samples = np.array(sorted(samples.unique()))
    n_samples = len(unique_samples)

    # -------- 3. collect per-sample matrices & centroids ----------
    sample_mats = {}
    sample_means = []

    for s in unique_samples:
        mask = (samples == s).values
        idx_all = np.where(mask)[0]
        if idx_all.size == 0:
            raise ValueError(f"No cells found for sample {s!r}.")

        # Optional subsampling for energy distance
        idx = idx_all
        if max_cells_per_sample is not None and idx.size > max_cells_per_sample:
            idx = rng.choice(idx, size=max_cells_per_sample, replace=False)

        X_sub = X[idx]

        # to dense
        if sp.issparse(X_sub):
            X_sub = X_sub.toarray()
        else:
            X_sub = np.asarray(X_sub)

        sample_mats[s] = X_sub

        # centroid for cosine distance (use *all* cells of that sample)
        X_all = X[idx_all]
        if sp.issparse(X_all):
            mean_vec = np.asarray(X_all.mean(axis=0)).ravel()
        else:
            mean_vec = np.asarray(X_all).mean(axis=0)
        sample_means.append(mean_vec)

    sample_means = np.vstack(sample_means)  # (n_samples, n_genes)

    # -------- 4. cosine distance between sample centroids ----------
    cos_mat = cosine_distances(sample_means, sample_means)
    cos_df = pd.DataFrame(cos_mat, index=unique_samples, columns=unique_samples)

    # -------- 5. multivariate energy distance between cell distributions ----------
    ed_mat = np.zeros((n_samples, n_samples), dtype=float)

    for i in range(n_samples):
        s_i = unique_samples[i]
        Xi = sample_mats[s_i]
        for j in range(i, n_samples):
            s_j = unique_samples[j]
            Xj = sample_mats[s_j]

            ed = energy_distance_multivariate(Xi, Xj, metric=metric)
            ed_mat[i, j] = ed
            ed_mat[j, i] = ed

    ed_df = pd.DataFrame(ed_mat, index=unique_samples, columns=unique_samples)

    # -------- 6. within-sample energy distances ----------
    # estimate via random split of each sample into two halves
    ed_within = []
    for s in unique_samples:
        Xi = sample_mats[s]
        m = Xi.shape[0]
        if m < 2:
            # cannot split; define as NaN
            ed_within.append(np.nan)
            continue

        vals = []
        for _ in range(within_reps):
            perm = rng.permutation(m)
            half = m // 2
            idx1 = perm[:half]
            idx2 = perm[half:]
            # if odd, second group has one more cell
            X1 = Xi[idx1]
            X2 = Xi[idx2]
            vals.append(energy_distance_multivariate(X1, X2, metric=metric))
        ed_within.append(float(np.mean(vals)))

    ed_within_df = pd.DataFrame(
        {"energy_within": ed_within}, index=unique_samples
    )

    # -------- 7. permutation test for between-sample energy distances ----------
    ed_pval_df = None
    if n_perm > 0:
        pvals = np.ones((n_samples, n_samples), dtype=float)

        for i in range(n_samples):
            s_i = unique_samples[i]
            Xi = sample_mats[s_i]
            n_i = Xi.shape[0]

            for j in range(i + 1, n_samples):
                s_j = unique_samples[j]
                Xj = sample_mats[s_j]
                n_j = Xj.shape[0]

                obs_ed = ed_mat[i, j]

                # pool cells and permute
                pool = np.vstack([Xi, Xj])
                N = pool.shape[0]
                if n_i + n_j != N:
                    raise RuntimeError("Size mismatch in permutation pooling.")

                perm_eds = []
                for _ in range(n_perm):
                    perm_idx = rng.permutation(N)
                    grpA_idx = perm_idx[:n_i]
                    grpB_idx = perm_idx[n_i:]
                    grpA = pool[grpA_idx]
                    grpB = pool[grpB_idx]
                    perm_eds.append(
                        energy_distance_multivariate(grpA, grpB, metric=metric)
                    )

                perm_eds = np.asarray(perm_eds)
                # one-sided: are groups more different than random?
                # p = P(ED_perm >= ED_obs)
                p = (1.0 + np.sum(perm_eds >= obs_ed)) / (n_perm + 1.0)
                pvals[i, j] = p
                pvals[j, i] = p

        np.fill_diagonal(pvals, 0.0)
        ed_pval_df = pd.DataFrame(pvals, index=unique_samples, columns=unique_samples)

    # -------- 8. save ----------
    cos_path = f"{out_prefix}_cosine.csv"
    ed_path  = f"{out_prefix}_energy.csv"
    ed_within_path = f"{out_prefix}_energy_within.csv"

    cos_df.to_csv(cos_path)
    ed_df.to_csv(ed_path)
    ed_within_df.to_csv(ed_within_path)

    print(f"Saved cosine distance matrix to: {cos_path}")
    print(f"Saved energy distance matrix to: {ed_path}")
    print(f"Saved within-sample energy distances to: {ed_within_path}")

    if ed_pval_df is not None:
        ed_pval_path = f"{out_prefix}_energy_pvals.csv"
        ed_pval_df.to_csv(ed_pval_path)
        print(f"Saved permutation p-values for energy distances to: {ed_pval_path}")

    return cos_df, ed_df, ed_within_df, ed_pval_df


In [ ]:
cos_df, ed_df, ed_within_df, ed_pval_df =compute_sample_distance_matrices_multivariate(
    adata_all,
    sample_key = "sample",
    out_prefix = "sample_distances_v2",
    max_cells_per_sample= 10000,
    within_reps = 10,
    n_perm= 100)
